# ASR Live Benchmark for PoliMillionaire

This notebook benchmarks speech recognition only. It does not load V5 indexes, rerankers, or LLMs.

Each ASR model has its own run cell. The shared CSV log is append-only and has a fixed schema. Whisper-family models write one candidate row per beam and one extra selected row.


In [ ]:
# Minimal setup for live ASR benchmarking.
import os, sys, json, time, random, gc, math, re
from pathlib import Path

import numpy as np
import pandas as pd

IN_COLAB = os.path.exists('/content') and not os.path.exists('/kaggle')
IN_KAGGLE = os.path.exists('/kaggle')
WORK_ROOT = Path('/kaggle/working') if IN_KAGGLE else (Path('/content/asr_benchmark_runtime') if IN_COLAB else Path.cwd() / '.runtime')
LOG_DIR = WORK_ROOT / 'logs'
AUDIO_ROOT = WORK_ROOT / 'asr_audio'
LOG_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_ROOT.mkdir(parents=True, exist_ok=True)

API_URL = 'http://131.175.15.22:51111/'
USERNAME_SECRET_NAME = 'USERNAME'
PASSWORD_SECRET_NAME = 'PASSWORD'
USERNAME = None
PASSWORD = None
API_REQUEST_TIMEOUT = 30
AUDIO_FETCH_RETRIES = 4
AUDIO_FETCH_RETRY_SLEEP = 2.5

ASR_BENCH_RUN_ID = time.strftime('%Y%m%d_%H%M%S_asr_live')
ASR_LIVE_LOG_CSV = LOG_DIR / 'asr_live_model_benchmark.csv'

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

print('WORK_ROOT:', WORK_ROOT)
print('AUDIO_ROOT:', AUDIO_ROOT)
print('ASR_LIVE_LOG_CSV:', ASR_LIVE_LOG_CSV)


In [ ]:
# Locate and import the official PoliMillionaire client.
def add_api_client_to_path():
    candidates = []
    if IN_KAGGLE:
        roots = [Path('/kaggle/input')]
        for root in roots:
            if root.exists():
                candidates.extend(root.rglob('NLP_assignment_api_client'))
    candidates.extend([
        Path.cwd() / 'api_client' / 'NLP_assignment_api_client',
        Path.cwd().parents[0] / 'api_client' / 'NLP_assignment_api_client' if len(Path.cwd().parents) > 0 else Path(''),
    ])
    for p in candidates:
        if p and (p / 'millionaire_client').exists():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            print('Using API client:', p)
            return p
    raise FileNotFoundError('Could not find NLP_assignment_api_client/millionaire_client.')

API_CLIENT_DIR = add_api_client_to_path()

from millionaire_client import MillionaireClient

def read_secret(secret_name):
    if not secret_name:
        return None
    if secret_name in os.environ and os.environ[secret_name]:
        return os.environ[secret_name]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(secret_name)
    except Exception:
        return None

def setup_client():
    username = USERNAME or read_secret(USERNAME_SECRET_NAME)
    password = PASSWORD or read_secret(PASSWORD_SECRET_NAME)
    if not username or not password:
        raise ValueError('Set USERNAME/PASSWORD or Kaggle secrets USERNAME/PASSWORD.')
    client = MillionaireClient(API_URL, timeout=API_REQUEST_TIMEOUT)
    client.login(username, password)
    return client

def get_competition_names(client):
    comps = client.competitions.list_all()
    for comp in comps:
        print(comp.id, comp.name, getattr(comp, 'max_levels', None))
    return {comp.id: comp.name for comp in comps}

client = setup_client()
competition_names = get_competition_names(client)


In [ ]:
# Live benchmark settings. Run this cell before a model-specific run cell.
# None means all categories returned by the API. For smoke tests use {'Ancient History and Politics'}.
BENCH_COMPETITIONS = None
BENCH_ATTEMPTS_PER_CATEGORY = 15
BENCH_MAX_QUESTIONS_PER_ATTEMPT = 15
BENCH_WAIT_BETWEEN_ATTEMPTS_SEC = 0.5
BENCH_API_THROTTLE_SEC = 0.5
BENCH_START_GAME_RETRIES = 3
BENCH_START_GAME_RETRY_SLEEP = 2.0
BENCH_SUBMIT_RANDOM_ANSWER = True

WHISPER_BENCH_BEAMS = [1, 3, 5]
SINGLE_PASS_BEAMS = [1]
SELECTED_ROW_RULE = 'best_mean_similarity_if_available_else_preferred_beam'
DEFAULT_SELECTED_BEAM = 1
SELECTED_BEAM_BY_MODEL = {
    'distil_medium_en': 1,
    'whisper_medium_en': 3,
    'whisper_large_v3_turbo': 3,
}

ASR_LIVE_LOG_CSV = LOG_DIR / 'asr_live_model_benchmark.csv'
LIVE_CLIPS = ['question', 'option_A', 'option_B', 'option_C', 'option_D']

LIVE_COLUMNS = [
    'bench_run_id', 'row_created_at', 'model_key', 'model_id', 'backend', 'device', 'dtype',
    'beam', 'beam_role', 'selected_from_beam', 'selected_rule', 'supports_beams', 'load_seconds_for_model',
    'competition_id', 'competition_name', 'attempt_number', 'question_index', 'session_id',
    'question_id', 'question_level', 'api_question_text',
    'api_option_A_id', 'api_option_A_text', 'api_option_B_id', 'api_option_B_text',
    'api_option_C_id', 'api_option_C_text', 'api_option_D_id', 'api_option_D_text',
    'question_audio_path', 'option_A_audio_path', 'option_B_audio_path', 'option_C_audio_path', 'option_D_audio_path',
    'question_audio_bytes', 'option_A_audio_bytes', 'option_B_audio_bytes', 'option_C_audio_bytes', 'option_D_audio_bytes',
    'question_audio_duration_seconds', 'option_A_audio_duration_seconds', 'option_B_audio_duration_seconds',
    'option_C_audio_duration_seconds', 'option_D_audio_duration_seconds',
    'fetch_question_seconds', 'fetch_option_A_seconds', 'fetch_option_B_seconds', 'fetch_option_C_seconds', 'fetch_option_D_seconds',
    'time_remaining_after_audio', 'random_option_id', 'answer_latency_seconds', 'random_answer_correct',
    'random_answer_timed_out', 'random_answer_game_over', 'earned_amount',
    'question_transcript', 'option_A_transcript', 'option_B_transcript', 'option_C_transcript', 'option_D_transcript',
    'question_asr_seconds', 'option_A_asr_seconds', 'option_B_asr_seconds', 'option_C_asr_seconds', 'option_D_asr_seconds',
    'total_asr_seconds', 'question_similarity', 'option_A_similarity', 'option_B_similarity', 'option_C_similarity',
    'option_D_similarity', 'mean_similarity', 'transcription_errors_json', 'error_stage', 'error_message',
]

def append_live_row(row, path=ASR_LIVE_LOG_CSV):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fixed = {col: row.get(col) for col in LIVE_COLUMNS}
    pd.DataFrame([fixed], columns=LIVE_COLUMNS).to_csv(
        path, mode='a', header=not path.exists() or path.stat().st_size == 0, index=False
    )

def normalize_for_eval(text):
    text = str(text or '').lower()
    text = re.sub(r'[^a-z0-9]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def text_similarity(pred, ref):
    from difflib import SequenceMatcher
    pred_n, ref_n = normalize_for_eval(pred), normalize_for_eval(ref)
    if not ref_n:
        return None
    if not pred_n:
        return 0.0
    return round(SequenceMatcher(None, pred_n, ref_n).ratio(), 4)

def selected_competitions_for_live(selected=BENCH_COMPETITIONS):
    items = list(competition_names.items())
    if selected is None:
        return items
    selected_set = set(selected)
    return [(cid, name) for cid, name in items if cid in selected_set or str(cid) in selected_set or name in selected_set]

def live_option_rows(question):
    rows = []
    for opt in list(getattr(question, 'options', []) or [])[:4]:
        rows.append((int(getattr(opt, 'id')), getattr(opt, 'text', None)))
    while len(rows) < 4:
        rows.append((None, None))
    return rows

def save_live_audio(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(data)
    return str(path)

def fmt_seconds(value):
    if value is None:
        return 'NA'
    try:
        if pd.isna(value):
            return 'NA'
        return f'{float(value):.2f}s'
    except Exception:
        return 'NA'

print('Live benchmark log:', ASR_LIVE_LOG_CSV)


In [ ]:
# Live benchmark model configs.
ASR_DEVICE = 'cuda:1'
ASR_DTYPE = 'float16'
ASR_PROMPT = (
    'Multiple choice quiz. Terms may include Achaeans, Argives, Danaans, Panhellenes, '
    'Ahhiyawa, Tawagalawa, Wilusa, Madduwatta, Ephors, Sparta, Carthage, '
    'Antikythera, Consecratio, Augurium, Acropolis, Illyrians, Pelasgians.'
)

LIVE_MODEL_CONFIGS = [
    {'key': 'distil_medium_en', 'model_id': 'distil-whisper/distil-medium.en', 'backend': 'whisper_seq2seq',
     'device': ASR_DEVICE, 'dtype': ASR_DTYPE, 'max_length': 448, 'beams': WHISPER_BENCH_BEAMS,
     'supports_beams': True, 'prompt': ASR_PROMPT},
    {'key': 'whisper_medium_en', 'model_id': 'openai/whisper-medium.en', 'backend': 'whisper_seq2seq',
     'device': ASR_DEVICE, 'dtype': ASR_DTYPE, 'max_length': 448, 'beams': WHISPER_BENCH_BEAMS,
     'supports_beams': True, 'prompt': ASR_PROMPT},
    {'key': 'whisper_large_v3_turbo', 'model_id': 'openai/whisper-large-v3-turbo', 'backend': 'whisper_seq2seq',
     'device': ASR_DEVICE, 'dtype': ASR_DTYPE, 'max_length': 448, 'beams': WHISPER_BENCH_BEAMS,
     'supports_beams': True, 'prompt': ASR_PROMPT},
    {'key': 'parakeet_tdt_0_6b_v3', 'model_id': 'nvidia/parakeet-tdt-0.6b-v3', 'backend': 'hf_asr_pipeline',
     'device': ASR_DEVICE, 'dtype': ASR_DTYPE, 'max_new_tokens': 256,
     'beams': SINGLE_PASS_BEAMS, 'supports_beams': False,
     'install_transformers_from_source': True},
    {'key': 'parakeet_tdt_0_6b_v2', 'model_id': 'nvidia/parakeet-tdt-0.6b-v2', 'backend': 'nemo_asr',
     'device': ASR_DEVICE, 'dtype': ASR_DTYPE, 'beams': SINGLE_PASS_BEAMS, 'supports_beams': False,
     'transcribe_kwargs': {}},
    {'key': 'canary_1b_v2', 'model_id': 'nvidia/canary-1b-v2', 'backend': 'nemo_asr',
     'device': ASR_DEVICE, 'dtype': ASR_DTYPE, 'beams': SINGLE_PASS_BEAMS, 'supports_beams': False,
     'transcribe_kwargs': {'source_lang': 'en', 'target_lang': 'en'}},
    {'key': 'nemotron_streaming_0_6b', 'model_id': 'nvidia/nemotron-speech-streaming-en-0.6b', 'backend': 'nemo_asr',
     'device': ASR_DEVICE, 'dtype': ASR_DTYPE, 'beams': SINGLE_PASS_BEAMS, 'supports_beams': False,
     'transcribe_kwargs': {}},
    {'key': 'qwen3_asr_1_7b', 'model_id': 'Qwen/Qwen3-ASR-1.7B', 'backend': 'qwen_asr',
     'device': ASR_DEVICE, 'dtype': 'bfloat16', 'max_new_tokens': 256, 'max_inference_batch_size': 1,
     'beams': SINGLE_PASS_BEAMS, 'supports_beams': False},
    {'key': 'wav2vec2_conformer_rope_large_960h_ft', 'model_id': 'facebook/wav2vec2-conformer-rope-large-960h-ft',
     'backend': 'hf_asr_pipeline', 'device': ASR_DEVICE, 'dtype': ASR_DTYPE,
     'beams': SINGLE_PASS_BEAMS, 'supports_beams': False,
     'force_pypi_transformers': True},
]

LIVE_CONFIG_BY_KEY = {cfg['key']: cfg for cfg in LIVE_MODEL_CONFIGS}
print('Live model configs:', list(LIVE_CONFIG_BY_KEY))


In [ ]:
# Live ASR backend implementations.
HF_HUB_COMPAT_SPEC = 'huggingface_hub>=0.34.0'


def clear_hf_import_cache():
    import importlib
    import sys
    for module_name in list(sys.modules):
        if module_name == 'huggingface_hub' or module_name.startswith('huggingface_hub.'):
            del sys.modules[module_name]
        elif module_name == 'transformers' or module_name.startswith('transformers.'):
            del sys.modules[module_name]
    importlib.invalidate_caches()


def install_base_asr_deps(force_pypi_transformers=False):
    import subprocess
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', '-U']
    if force_pypi_transformers:
        cmd += ['--force-reinstall', '--no-cache-dir']
    cmd += ['transformers', 'accelerate', HF_HUB_COMPAT_SPEC, 'soundfile', 'scipy', 'librosa']
    subprocess.check_call(cmd)
    clear_hf_import_cache()


def maybe_install_transformers_source():
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', HF_HUB_COMPAT_SPEC, 'git+https://github.com/huggingface/transformers'])
    clear_hf_import_cache()


def install_nemo_asr_deps():
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'nemo_toolkit[asr]', HF_HUB_COMPAT_SPEC])
    clear_hf_import_cache()


def decode_wav(path, target_sr=16000):
    import soundfile as sf
    import librosa
    audio, sr = sf.read(str(path), dtype='float32', always_2d=False)
    if getattr(audio, 'ndim', 1) > 1:
        audio = audio.mean(axis=1)
    audio = np.asarray(audio, dtype=np.float32)
    if sr != target_sr:
        audio = librosa.resample(audio, orig_sr=float(sr), target_sr=float(target_sr))
        sr = target_sr
    return audio, int(sr)


def audio_duration_seconds(path):
    import soundfile as sf
    info = sf.info(str(path))
    return float(info.frames) / float(info.samplerate) if info.samplerate else None


def clean_transcript(text):
    text = str(text or '').strip()
    text = re.sub(r'\s+', ' ', text)
    return text


def extract_live_text(output):
    if not output and output != 0:
        return ''
    if isinstance(output, list):
        return extract_live_text(output[0])
    if isinstance(output, tuple):
        return extract_live_text(output[0])
    if isinstance(output, dict):
        return extract_live_text(output.get('text', ''))
    if hasattr(output, 'text'):
        return extract_live_text(output.text)
    return clean_transcript(output)


def dtype_from_name(name):
    import torch
    low = str(name).lower()
    if low in ('float16', 'fp16'):
        return torch.float16
    if low in ('bfloat16', 'bf16'):
        return torch.bfloat16
    return torch.float32


class WhisperSeq2SeqRunner:
    def __init__(self, cfg):
        install_base_asr_deps()
        import torch
        from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
        self.cfg = cfg
        requested_device = cfg.get('device', 'cuda:0')
        self.device = torch.device(requested_device if torch.cuda.is_available() else 'cpu')
        self.dtype = dtype_from_name(cfg.get('dtype', 'float16')) if self.device.type == 'cuda' else torch.float32
        self.processor = AutoProcessor.from_pretrained(cfg['model_id'])
        self.model = AutoModelForSpeechSeq2Seq.from_pretrained(
            cfg['model_id'],
            torch_dtype=self.dtype,
            low_cpu_mem_usage=True,
            use_safetensors=True,
        ).to(self.device)
        self.model.eval()
        if hasattr(self.model, 'generation_config') and hasattr(self.model.generation_config, 'return_timestamps'):
            self.model.generation_config.return_timestamps = False

    def _generate_and_decode(self, input_features, kwargs):
        import torch
        last_type_error = None
        for extra in ({'language': 'english', 'task': 'transcribe'}, {}):
            try:
                with torch.inference_mode():
                    pred = self.model.generate(input_features, **kwargs, **extra)
                return clean_transcript(self.processor.batch_decode(pred, skip_special_tokens=True)[0])
            except (TypeError, ValueError) as exc:
                last_type_error = exc
                continue
        if last_type_error is not None:
            with torch.inference_mode():
                pred = self.model.generate(input_features, **kwargs)
            return clean_transcript(self.processor.batch_decode(pred, skip_special_tokens=True)[0])

    def transcribe(self, path, beam=1):
        audio, sr = decode_wav(path, target_sr=16000)
        inputs = self.processor(audio, sampling_rate=sr, return_tensors='pt')
        input_features = inputs.input_features.to(device=self.device, dtype=self.dtype)
        kwargs = {
            'max_length': int(self.cfg.get('max_length', 448)),
            'return_timestamps': False,
            'num_beams': max(1, int(beam or 1)),
        }
        prompt = self.cfg.get('prompt')
        if prompt:
            try:
                kwargs['prompt_ids'] = self.processor.get_prompt_ids(prompt, return_tensors='pt').to(self.device)
            except Exception:
                pass
        try:
            return self._generate_and_decode(input_features, kwargs)
        except Exception:
            if 'prompt_ids' not in kwargs:
                raise
            kwargs = dict(kwargs)
            kwargs.pop('prompt_ids', None)
            return self._generate_and_decode(input_features, kwargs)

    def close(self):
        try:
            self.model.cpu()
        except Exception:
            pass
        del self.model
        del self.processor


class HfAsrPipelineRunner:
    def __init__(self, cfg):
        if cfg.get('install_transformers_from_source'):
            maybe_install_transformers_source()
        else:
            install_base_asr_deps(force_pypi_transformers=bool(cfg.get('force_pypi_transformers')))
        import torch
        from transformers import pipeline
        self.cfg = cfg
        device = cfg.get('device', 'cuda:0')
        if torch.cuda.is_available() and str(device).startswith('cuda'):
            device_arg = int(str(device).split(':')[1]) if ':' in str(device) else 0
            dtype = dtype_from_name(cfg.get('dtype', 'float16'))
        else:
            device_arg = -1
            dtype = torch.float32
        self.pipe = pipeline('automatic-speech-recognition', model=cfg['model_id'], device=device_arg, torch_dtype=dtype)

    def transcribe(self, path, beam=1):
        generate_kwargs = dict(self.cfg.get('generate_kwargs') or {})
        if self.cfg.get('max_new_tokens') is not None:
            generate_kwargs['max_new_tokens'] = int(self.cfg['max_new_tokens'])
        if self.cfg.get('supports_beams'):
            generate_kwargs['num_beams'] = max(1, int(beam or 1))
        if generate_kwargs:
            try:
                import warnings
                with warnings.catch_warnings():
                    warnings.filterwarnings('ignore', message=r'Using the model-agnostic default `max_length`.*', category=UserWarning)
                    out = self.pipe(str(path), generate_kwargs=generate_kwargs)
            except TypeError:
                out = self.pipe(str(path))
        else:
            out = self.pipe(str(path))
        return extract_live_text(out)

    def close(self):
        try:
            if hasattr(self.pipe, 'model'):
                self.pipe.model.cpu()
        except Exception:
            pass
        del self.pipe


class NemoAsrRunner:
    def __init__(self, cfg):
        install_base_asr_deps()
        install_nemo_asr_deps()
        import torch
        import nemo.collections.asr as nemo_asr
        self.cfg = cfg
        self.device = cfg.get('device', 'cuda:0') if torch.cuda.is_available() else 'cpu'
        self.model = nemo_asr.models.ASRModel.from_pretrained(model_name=cfg['model_id'])
        try:
            self.model = self.model.to(self.device)
            self.model.eval()
        except Exception:
            pass

    def transcribe(self, path, beam=1):
        kwargs = dict(self.cfg.get('transcribe_kwargs') or {})
        last_exc = None
        for key in [None, 'paths2audio_files', 'audio']:
            try:
                if key:
                    return extract_live_text(self.model.transcribe(**{key: [str(path)]}, **kwargs))
                return extract_live_text(self.model.transcribe([str(path)], **kwargs))
            except TypeError as exc:
                last_exc = exc
                continue
        try:
            return extract_live_text(self.model.transcribe([str(path)]))
        except Exception:
            if last_exc:
                raise last_exc
            raise

    def close(self):
        try:
            self.model.cpu()
        except Exception:
            pass
        del self.model


class QwenAsrRunner:
    def __init__(self, cfg):
        import subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'qwen-asr'])
        clear_hf_import_cache()
        pip_check = subprocess.run([sys.executable, '-m', 'pip', 'check'], text=True, capture_output=True)
        if pip_check.returncode != 0:
            print('pip check reported dependency conflicts after qwen-asr install:')
            print((pip_check.stdout + pip_check.stderr).strip())
        from qwen_asr import Qwen3ASRModel
        self.cfg = cfg
        dtype = dtype_from_name(cfg.get('dtype', 'bfloat16'))
        self.model = Qwen3ASRModel.from_pretrained(
            cfg['model_id'],
            dtype=dtype,
            device_map=cfg.get('device', 'cuda:0'),
            max_inference_batch_size=int(cfg.get('max_inference_batch_size', 1)),
            max_new_tokens=int(cfg.get('max_new_tokens', 256)),
        )

    def transcribe(self, path, beam=1):
        from transformers.utils import logging as hf_logging
        old_verbosity = hf_logging.get_verbosity()
        hf_logging.set_verbosity_error()
        try:
            try:
                results = self.model.transcribe(audio=str(path), language='English')
            except TypeError:
                results = self.model.transcribe(str(path))
        finally:
            hf_logging.set_verbosity(old_verbosity)
        return extract_live_text(results)

    def close(self):
        try:
            if hasattr(self.model, 'model') and hasattr(self.model.model, 'cpu'):
                self.model.model.cpu()
            elif hasattr(self.model, 'cpu'):
                self.model.cpu()
        except Exception:
            pass
        del self.model


def make_live_runner(cfg):
    backend = cfg['backend']
    if backend == 'whisper_seq2seq':
        return WhisperSeq2SeqRunner(cfg)
    if backend == 'hf_asr_pipeline':
        return HfAsrPipelineRunner(cfg)
    if backend == 'nemo_asr':
        return NemoAsrRunner(cfg)
    if backend == 'qwen_asr':
        return QwenAsrRunner(cfg)
    raise ValueError(f'Unknown backend: {backend}')


def live_transcribe(runner, cfg, path, beam):
    try:
        return runner.transcribe(path, beam=beam)
    except TypeError:
        return runner.transcribe(path)


def close_runner(runner):
    try:
        runner.close()
    finally:
        gc.collect()
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
        except Exception:
            pass


In [1]:
# Live benchmark runner: one append-only row per beam plus one selected row.
def throttle_live_api():
    delay = float(globals().get('BENCH_API_THROTTLE_SEC', 0.0) or 0.0)
    if delay > 0:
        time.sleep(delay)


def start_speech_game_with_retry(client, comp_id):
    last_exc = None
    for attempt in range(1, int(BENCH_START_GAME_RETRIES) + 1):
        try:
            game = client.game.start(competition_id=comp_id, mode='speech')
            throttle_live_api()
            return game
        except KeyboardInterrupt:
            raise
        except Exception as exc:
            last_exc = exc
            print(f'  start_game failed attempt {attempt}/{BENCH_START_GAME_RETRIES}: {type(exc).__name__}: {exc}', flush=True)
            if attempt < int(BENCH_START_GAME_RETRIES):
                time.sleep(BENCH_START_GAME_RETRY_SLEEP)
    raise RuntimeError(f'start_game failed after {BENCH_START_GAME_RETRIES} attempts: {last_exc!r}')


def fetch_audio_with_retry(fetch_fn, label):
    last_exc = None
    for attempt in range(1, AUDIO_FETCH_RETRIES + 1):
        started = time.time()
        try:
            data = fetch_fn()
            throttle_live_api()
            return data, time.time() - started
        except KeyboardInterrupt:
            raise
        except Exception as exc:
            last_exc = exc
            print(f'  fetch {label} failed attempt {attempt}/{AUDIO_FETCH_RETRIES}: {type(exc).__name__}: {exc}', flush=True)
            time.sleep(AUDIO_FETCH_RETRY_SLEEP)
    raise RuntimeError(f'fetch {label} failed after {AUDIO_FETCH_RETRIES} attempts: {last_exc!r}')

def fetch_live_bundle(game, comp_id, comp_name, attempt_number, question_index, model_key):
    question = game.current_question
    if question is None:
        raise RuntimeError('No current question in active speech game.')
    session_id = getattr(game, 'session_id', None)
    qid = getattr(question, 'id', None)
    level = getattr(question, 'level', None)
    base = AUDIO_ROOT / ASR_BENCH_RUN_ID / model_key / f'comp_{comp_id}' / f'attempt_{attempt_number:03d}' / f'session_{session_id}_q{question_index:02d}_qid_{qid}'

    paths, fetch_seconds, sizes = {}, {}, {}
    data, elapsed = fetch_audio_with_retry(game.fetch_audio_question, 'question')
    fetch_seconds['question'] = elapsed
    paths['question'] = save_live_audio(base / 'question.wav', data)
    sizes['question'] = len(data)

    for idx in range(4):
        label = f'option_{chr(65 + idx)}'
        data, elapsed = fetch_audio_with_retry(game.fetch_audio_option_next, label)
        fetch_seconds[label] = elapsed
        paths[label] = save_live_audio(base / f'{label}.wav', data)
        sizes[label] = len(data)

    try:
        game.refresh_state()
        throttle_live_api()
    except Exception:
        pass

    random_option_id, answer_latency_seconds, answer_error = None, None, None
    result_fields = {'random_answer_correct': None, 'random_answer_timed_out': None, 'random_answer_game_over': None, 'earned_amount': None}
    if BENCH_SUBMIT_RANDOM_ANSWER:
        opts = list(getattr(question, 'options', []) or [])
        if opts:
            random_option_id = int(random.choice(opts).id)
            try:
                started = time.time()
                result = game.answer(random_option_id)
                throttle_live_api()
                answer_latency_seconds = time.time() - started
                result_fields = {
                    'random_answer_correct': getattr(result, 'correct', None),
                    'random_answer_timed_out': getattr(result, 'timed_out', None),
                    'random_answer_game_over': getattr(result, 'game_over', None),
                    'earned_amount': getattr(result, 'earned_amount', None),
                }
            except Exception as exc:
                answer_error = repr(exc)

    opt_rows = live_option_rows(question)
    return {
        'competition_id': comp_id, 'competition_name': comp_name, 'attempt_number': attempt_number,
        'question_index': question_index, 'session_id': session_id, 'question_id': qid, 'question_level': level,
        'api_question_text': getattr(question, 'text', None),
        'api_option_A_id': opt_rows[0][0], 'api_option_A_text': opt_rows[0][1],
        'api_option_B_id': opt_rows[1][0], 'api_option_B_text': opt_rows[1][1],
        'api_option_C_id': opt_rows[2][0], 'api_option_C_text': opt_rows[2][1],
        'api_option_D_id': opt_rows[3][0], 'api_option_D_text': opt_rows[3][1],
        'paths': paths, 'fetch_seconds': fetch_seconds, 'sizes': sizes,
        'time_remaining_after_audio': getattr(game, 'time_remaining', None),
        'random_option_id': random_option_id, 'answer_latency_seconds': answer_latency_seconds,
        'answer_error': answer_error, **result_fields,
    }

def transcribe_live_bundle(runner, cfg, bundle, beam, load_seconds):
    row = {
        'bench_run_id': ASR_BENCH_RUN_ID, 'row_created_at': time.strftime('%Y-%m-%d %H:%M:%S'),
        'model_key': cfg['key'], 'model_id': cfg['model_id'], 'backend': cfg['backend'],
        'device': cfg.get('device'), 'dtype': cfg.get('dtype'), 'beam': int(beam or 1),
        'beam_role': 'candidate', 'selected_from_beam': None, 'selected_rule': None,
        'supports_beams': bool(cfg.get('supports_beams')), 'load_seconds_for_model': load_seconds,
    }
    for key in [
        'competition_id', 'competition_name', 'attempt_number', 'question_index', 'session_id',
        'question_id', 'question_level', 'api_question_text',
        'api_option_A_id', 'api_option_A_text', 'api_option_B_id', 'api_option_B_text',
        'api_option_C_id', 'api_option_C_text', 'api_option_D_id', 'api_option_D_text',
        'time_remaining_after_audio', 'random_option_id', 'answer_latency_seconds',
        'random_answer_correct', 'random_answer_timed_out', 'random_answer_game_over', 'earned_amount',
    ]:
        row[key] = bundle.get(key)
    if bundle.get('answer_error'):
        row['error_stage'] = 'answer'
        row['error_message'] = bundle.get('answer_error')

    errors, total = {}, 0.0
    for label in LIVE_CLIPS:
        path = Path(bundle['paths'][label])
        row[f'{label}_audio_path'] = str(path)
        row[f'{label}_audio_bytes'] = bundle['sizes'].get(label)
        row[f'{label}_audio_duration_seconds'] = audio_duration_seconds(path)
        row[f'fetch_{label}_seconds'] = bundle['fetch_seconds'].get(label)
        try:
            started = time.time()
            row[f'{label}_transcript'] = live_transcribe(runner, cfg, path, beam)
            elapsed = time.time() - started
            row[f'{label}_asr_seconds'] = elapsed
            total += elapsed
        except Exception as exc:
            errors[label] = repr(exc)
            row[f'{label}_transcript'] = None
            row[f'{label}_asr_seconds'] = None

    row['total_asr_seconds'] = total if total else None
    row['question_similarity'] = text_similarity(row.get('question_transcript'), row.get('api_question_text'))
    sims = [row['question_similarity']]
    for letter in ['A', 'B', 'C', 'D']:
        sim = text_similarity(row.get(f'option_{letter}_transcript'), row.get(f'api_option_{letter}_text'))
        row[f'option_{letter}_similarity'] = sim
        sims.append(sim)
    valid_sims = [s for s in sims if s is not None and not pd.isna(s)]
    row['mean_similarity'] = round(sum(valid_sims) / len(valid_sims), 4) if valid_sims else None
    row['transcription_errors_json'] = json.dumps(errors, ensure_ascii=False) if errors else None
    if errors:
        row['error_stage'] = 'transcription'
        row['error_message'] = row['transcription_errors_json']
    return row

def make_selected_live_row(candidate_rows):
    if not candidate_rows:
        return None
    model_key = candidate_rows[0].get('model_key')
    preferred_beam = SELECTED_BEAM_BY_MODEL.get(model_key, DEFAULT_SELECTED_BEAM)
    has_any_similarity = any(
        row.get('mean_similarity') is not None and not pd.isna(row.get('mean_similarity'))
        for row in candidate_rows
    )
    def score(row):
        sec = row.get('total_asr_seconds')
        sec = 1e9 if sec is None or pd.isna(sec) else float(sec)
        has_error = 1 if row.get('error_message') else 0
        if has_any_similarity:
            sim = row.get('mean_similarity')
            sim = -1.0 if sim is None or pd.isna(sim) else float(sim)
            return (-has_error, sim, -sec)
        is_preferred = int(row.get('beam') == preferred_beam)
        return (-has_error, is_preferred, -sec)
    selected = max(candidate_rows, key=score).copy()
    selected['row_created_at'] = time.strftime('%Y-%m-%d %H:%M:%S')
    selected['selected_from_beam'] = selected.get('beam')
    selected['beam'] = 'selected'
    selected['beam_role'] = 'selected'
    selected['selected_rule'] = SELECTED_ROW_RULE
    return selected

def run_live_asr_benchmark_for_model(model_key, beams=None, competitions=None, attempts_per_category=None, max_questions_per_attempt=None):
    cfg = dict(LIVE_CONFIG_BY_KEY[model_key])
    beams = list(beams if beams is not None else cfg.get('beams', [1]))
    comps = selected_competitions_for_live(BENCH_COMPETITIONS if competitions is None else competitions)
    attempts = int(attempts_per_category or BENCH_ATTEMPTS_PER_CATEGORY)
    max_q = int(max_questions_per_attempt or BENCH_MAX_QUESTIONS_PER_ATTEMPT)
    runner, rows = None, []
    load_started = time.time()
    print(f"\n### Loading {cfg['key']} | {cfg['model_id']} | {cfg['backend']} on {cfg.get('device')} ###", flush=True)
    try:
        runner = make_live_runner(cfg)
        load_seconds = time.time() - load_started
        print(f'Loaded in {load_seconds:.1f}s. Beams: {beams}. Log: {ASR_LIVE_LOG_CSV}', flush=True)
        for comp_id, comp_name in comps:
            for attempt in range(1, attempts + 1):
                print(f"\n[{cfg['key']}] {comp_name} attempt {attempt}/{attempts}", flush=True)
                try:
                    game = start_speech_game_with_retry(client, comp_id)
                except Exception as exc:
                    row = {'bench_run_id': ASR_BENCH_RUN_ID, 'row_created_at': time.strftime('%Y-%m-%d %H:%M:%S'),
                           'model_key': cfg['key'], 'model_id': cfg['model_id'], 'backend': cfg['backend'],
                           'device': cfg.get('device'), 'dtype': cfg.get('dtype'), 'beam_role': 'error',
                           'competition_id': comp_id, 'competition_name': comp_name, 'attempt_number': attempt,
                           'supports_beams': bool(cfg.get('supports_beams')), 'load_seconds_for_model': load_seconds,
                           'error_stage': 'start_game', 'error_message': repr(exc)}
                    append_live_row(row)
                    rows.append(row)
                    print(f'  start_game error: {exc}', flush=True)
                    if attempt < attempts:
                        time.sleep(BENCH_WAIT_BETWEEN_ATTEMPTS_SEC)
                    continue

                qidx = 0
                while getattr(game, 'in_progress', False) and qidx < max_q:
                    qidx += 1
                    try:
                        bundle = fetch_live_bundle(game, comp_id, comp_name, attempt, qidx, cfg['key'])
                    except Exception as exc:
                        row = {'bench_run_id': ASR_BENCH_RUN_ID, 'row_created_at': time.strftime('%Y-%m-%d %H:%M:%S'),
                               'model_key': cfg['key'], 'model_id': cfg['model_id'], 'backend': cfg['backend'],
                               'device': cfg.get('device'), 'dtype': cfg.get('dtype'), 'beam_role': 'error',
                               'competition_id': comp_id, 'competition_name': comp_name, 'attempt_number': attempt,
                               'question_index': qidx, 'session_id': getattr(game, 'session_id', None),
                               'supports_beams': bool(cfg.get('supports_beams')), 'load_seconds_for_model': load_seconds,
                               'error_stage': 'fetch_audio', 'error_message': repr(exc)}
                        append_live_row(row)
                        rows.append(row)
                        print(f'  fetch_audio error: {exc}', flush=True)
                        break

                    candidate_rows = []
                    for beam in beams:
                        row = transcribe_live_bundle(runner, cfg, bundle, beam, load_seconds)
                        append_live_row(row)
                        rows.append(row)
                        candidate_rows.append(row)
                        if row.get('error_message'):
                            print(f"  qid={row.get('question_id')} beam={beam} ERROR {str(row.get('error_message'))[:260]}", flush=True)
                        else:
                            print(f"  qid={row.get('question_id')} beam={beam} mean_sim={row.get('mean_similarity')} asr={fmt_seconds(row.get('total_asr_seconds'))} | {str(row.get('question_transcript'))[:90]}", flush=True)
                    selected = make_selected_live_row(candidate_rows)
                    if selected is not None:
                        append_live_row(selected)
                        rows.append(selected)
                        print(f"  selected_from_beam={selected.get('selected_from_beam')} mean_sim={selected.get('mean_similarity')} asr={fmt_seconds(selected.get('total_asr_seconds'))}", flush=True)
                if attempt < attempts:
                    time.sleep(BENCH_WAIT_BETWEEN_ATTEMPTS_SEC)
    finally:
        if runner is not None:
            print(f"Freeing {cfg['key']} from GPU...", flush=True)
            close_runner(runner)
            print('GPU cache cleared.', flush=True)
    return pd.DataFrame(rows)


In [ ]:
# Run distil-whisper/distil-medium.en
# Appends to ASR_LIVE_LOG_CSV and frees GPU at the end.
distil_medium_en_live_df = run_live_asr_benchmark_for_model('distil_medium_en')
distil_medium_en_live_df.tail()


In [ ]:
# Run openai/whisper-medium.en
# Appends to ASR_LIVE_LOG_CSV and frees GPU at the end.
whisper_medium_en_live_df = run_live_asr_benchmark_for_model('whisper_medium_en')
whisper_medium_en_live_df.tail()


In [ ]:
# Run openai/whisper-large-v3-turbo
# Appends to ASR_LIVE_LOG_CSV and frees GPU at the end.
whisper_large_v3_turbo_live_df = run_live_asr_benchmark_for_model('whisper_large_v3_turbo')
whisper_large_v3_turbo_live_df.tail()


In [ ]:
# Run nvidia/parakeet-tdt-0.6b-v3
# Appends to ASR_LIVE_LOG_CSV and frees GPU at the end.
parakeet_tdt_0_6b_v3_live_df = run_live_asr_benchmark_for_model('parakeet_tdt_0_6b_v3')
parakeet_tdt_0_6b_v3_live_df.tail()


In [ ]:
# Run nvidia/parakeet-tdt-0.6b-v2
# Appends to ASR_LIVE_LOG_CSV and frees GPU at the end.
parakeet_tdt_0_6b_v2_live_df = run_live_asr_benchmark_for_model('parakeet_tdt_0_6b_v2')
parakeet_tdt_0_6b_v2_live_df.tail()


In [ ]:
# Run nvidia/canary-1b-v2
# Appends to ASR_LIVE_LOG_CSV and frees GPU at the end.
canary_1b_v2_live_df = run_live_asr_benchmark_for_model('canary_1b_v2')
canary_1b_v2_live_df.tail()


In [ ]:
# Run nvidia/nemotron-speech-streaming-en-0.6b
# Appends to ASR_LIVE_LOG_CSV and frees GPU at the end.
nemotron_streaming_0_6b_live_df = run_live_asr_benchmark_for_model('nemotron_streaming_0_6b')
nemotron_streaming_0_6b_live_df.tail()


In [ ]:
# Run Qwen/Qwen3-ASR-1.7B
# Appends to ASR_LIVE_LOG_CSV and frees GPU at the end.
qwen3_asr_1_7b_live_df = run_live_asr_benchmark_for_model('qwen3_asr_1_7b')
qwen3_asr_1_7b_live_df.tail()


In [ ]:
# Run facebook/wav2vec2-conformer-rope-large-960h-ft
# Appends to ASR_LIVE_LOG_CSV and frees GPU at the end.
wav2vec2_conformer_rope_large_960h_ft_live_df = run_live_asr_benchmark_for_model('wav2vec2_conformer_rope_large_960h_ft')
wav2vec2_conformer_rope_large_960h_ft_live_df.tail()


In [ ]:
# Summary for the live append-only benchmark log.
if ASR_LIVE_LOG_CSV.exists():
    df = pd.read_csv(ASR_LIVE_LOG_CSV)
    print('Log:', ASR_LIVE_LOG_CSV)
    print('Rows:', len(df))
    ok = df[df['error_message'].isna()].copy()
    if len(ok):
        display(
            ok.groupby(['model_key', 'beam_role', 'beam'], dropna=False)
              .agg(
                  rows=('mean_similarity', 'count'),
                  mean_similarity=('mean_similarity', 'mean'),
                  question_similarity=('question_similarity', 'mean'),
                  total_asr_seconds=('total_asr_seconds', 'mean'),
              )
              .reset_index()
              .sort_values(['beam_role', 'mean_similarity', 'total_asr_seconds'], ascending=[True, False, True])
        )
    err = df[df['error_message'].notna()]
    if len(err):
        display(err[['model_key', 'beam_role', 'beam', 'competition_name', 'attempt_number', 'question_id', 'error_stage', 'error_message']].tail(50))
    preview_cols = [
        'model_key', 'beam_role', 'beam', 'selected_from_beam', 'competition_name', 'question_id',
        'mean_similarity', 'total_asr_seconds', 'api_question_text', 'question_transcript',
        'api_option_A_text', 'option_A_transcript', 'api_option_B_text', 'option_B_transcript',
        'api_option_C_text', 'option_C_transcript', 'api_option_D_text', 'option_D_transcript',
    ]
    display(df[preview_cols].tail(80))
else:
    print('No live log yet:', ASR_LIVE_LOG_CSV)
